Data Cleaning

In [8]:
import sys
!{sys.executable} -m pip install pydub
!{sys.executable} -m pip install ffmpeg-downloader

In [9]:
#from google.colab import drive
from datetime import datetime as dt
import geopandas
from pydub import AudioSegment
import numpy as np
import zipfile
#drive.mount('/content/drive')

In [10]:
import pandas as pd

# The file is in Google Drive within 'Shared drives'.
file_path = '/Users/shloka/Documents/urban sonyc data/sonyc-washington-sq-data.csv'

# This try-except block handles the case where the file might not be found
# or if Drive is not yet mounted/authorized.
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from '{file_path}'.")
    print("Displaying the first 5 rows of the DataFrame:")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the file is in the correct location and the path is accurate.")
    print("If your data is in Google Drive, ensure you have mounted your Drive first and updated the 'file_path' variable accordingly.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully loaded data from '/Users/shloka/Documents/urban sonyc data/sonyc-washington-sq-data.csv'.
Displaying the first 5 rows of the DataFrame:


In [11]:
# Source - https://stackoverflow.com/a/63072225
# Posted by ipj, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-25, License - CC BY-SA 4.0

def ftod(x):
    if (x>=20) & (x<6):
        tod = 'night'
    elif (x>=6) & (x<12):
        tod = 'morning'
    elif (x>=12) & (x<18):
        tod = 'afternoon'
    else:
        tod = 'evening'
    return tod

In [12]:
# cleaned based on:
#'The anonymous ID of the annotator. If this value is positive, it is a citizen science volunteer from the Zooniverse platform. If it is negative,
# it is a SONYC team member. If it is 0, then it is the ground truth agreed-upon by the SONYC team.'

df_cleaned = df[df['split'] == 'test']
df_verified = df_cleaned[df['annotator_id'] == 0]

df_verified['date_time'] = df_verified['year'].astype(str) + '-' + \
                            df_verified['week'].astype(str) + '-' + \
                            df_verified['day'].astype(str) + '-' + \
                            df_verified['hour'].astype(str)

date_time_format = '%Y-%W-%w-%H'
df_verified['date_time'] = pd.to_datetime(df_verified['date_time'], format=date_time_format)

# https://stackoverflow.com/questions/63071619/how-to-categorize-timestamp-into-evening-in-pandas-dataframe
df_verified['time_of_day'] = df_verified['date_time'].dt.hour.map(ftod)

df_verified.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 81 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   split                                         311 non-null    object        
 1   sensor_id                                     311 non-null    int64         
 2   audio_filename                                311 non-null    object        
 3   annotator_id                                  311 non-null    int64         
 4   borough                                       311 non-null    int64         
 5   block                                         311 non-null    int64         
 6   latitude                                      311 non-null    float64       
 7   longitude                                     311 non-null    float64       
 8   year                                          311 non-null    int64   

In [13]:
df_verified.head()

,split,sensor_id,audio_filename,annotator_id,borough,block,latitude,longitude,year,week,...,2_machinery-impact_presence,3_non-machinery-impact_presence,4_powered-saw_presence,5_alert-signal_presence,6_music_presence,7_human-voice_presence,8_dog_presence,date_time,geometry,time_of_day
0,test,1,01_026900.wav,0,1,541,40.73033,-73.9987,2019,31,...,0,1,0,0,1,1,0,2019-08-07 19:00:00,POINT (-73.9987 40.73033),evening
1,test,1,01_026905.wav,0,1,541,40.73033,-73.9987,2019,51,...,0,1,0,0,0,0,1,2019-12-29 13:00:00,POINT (-73.9987 40.73033),afternoon
2,test,1,01_026976.wav,0,1,541,40.73033,-73.9987,2019,20,...,0,1,0,0,0,0,0,2019-05-22 14:00:00,POINT (-73.9987 40.73033),afternoon
3,test,1,01_026984.wav,0,1,541,40.73033,-73.9987,2019,23,...,0,0,0,1,0,0,1,2019-06-11 19:00:00,POINT (-73.9987 40.73033),evening
4,test,1,01_027018.wav,0,1,541,40.73033,-73.9987,2019,31,...,0,0,0,0,0,1,0,2019-08-07 18:00:00,POINT (-73.9987 40.73033),evening


In [14]:
with zipfile.ZipFile('/Volumes/Extreme SSD/3966543.zip', 'r') as zip_ref:
    zip_ref.extractall('/Volumes/Extreme SSD/unzipped_sounds')

In [17]:
import os
import shutil
import tarfile
import glob
from pathlib import Path

SOURCE_DIR   = "/Volumes/Extreme SSD/unzipped_sounds"                                   
DEST_DIR     = "/Volumes/Extreme SSD/audio_verified" 

target_files = set(df_verified['audio_filename'].tolist())
print(f"Looking for {len(target_files)} audio files...")

os.makedirs(DEST_DIR, exist_ok=True)
print(f"Destination folder ready: {DEST_DIR}")

Looking for 311 audio files...
Destination folder ready: /Volumes/Extreme SSD/audio_verified


In [18]:
STAGING_DIR = "./audio_staging"
os.makedirs(STAGING_DIR, exist_ok=True)

tar_files = glob.glob(os.path.join(SOURCE_DIR, "audio-*.tar.gz"))
print(f"Found {len(tar_files)} tar.gz archives to extract...")

for tar_path in tar_files:
    print(f"  Extracting: {tar_path}")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=STAGING_DIR)

print("Extraction complete.")

Found 19 tar.gz archives to extract...
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-1.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-5.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-6.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-7.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-8.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-9.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-4.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-3.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-2.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-0.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-10.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-11.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-12.tar.gz
  Extracting: /Volumes/Extreme SSD/unzipped_sounds/audio-13.tar.gz
  Extracting: /Volumes/Extreme SS

In [19]:
# 4. Find all .wav files in the staging area and copy matches to the SSD
all_wavs = glob.glob(os.path.join(STAGING_DIR, "**", "*.wav"), recursive=True)
print(f"Found {len(all_wavs)} total .wav files in staging area.")

copied, skipped = 0, 0

for wav_path in all_wavs:
    filename = os.path.basename(wav_path)
    if filename in target_files:
        dest_path = os.path.join(DEST_DIR, filename)
        shutil.copy2(wav_path, dest_path)
        copied += 1
    else:
        skipped += 1

print(f"\nCopied : {copied} files → {DEST_DIR}")
print(f"Skipped: {skipped} files (not in df_verified)")

Found 18510 total .wav files in staging area.

Copied : 311 files → /Volumes/Extreme SSD/audio_verified
Skipped: 18199 files (not in df_verified)


In [20]:
AUDIO_DIR = "/Volumes/Extreme SSD/audio_verified"

def get_average_volume(filename):
    try:
        path = os.path.join(AUDIO_DIR, filename)
        audio = AudioSegment.from_wav(path)
        samples = np.array(audio.get_array_of_samples())
        avg_volume = 20 * np.log10(np.sqrt(np.mean(samples**2)) + 1e-10)  # dBFS
        return round(avg_volume, 2)
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None

df_verified['avg_volume_db'] = df_verified['audio_filename'].apply(get_average_volume)

In [21]:
df_verified.head()

,split,sensor_id,audio_filename,annotator_id,borough,block,latitude,longitude,year,week,...,3_non-machinery-impact_presence,4_powered-saw_presence,5_alert-signal_presence,6_music_presence,7_human-voice_presence,8_dog_presence,date_time,geometry,time_of_day,avg_volume_db
0,test,1,01_026900.wav,0,1,541,40.73033,-73.9987,2019,31,...,1,0,0,1,1,0,2019-08-07 19:00:00,POINT (-73.9987 40.73033),evening,29.15
1,test,1,01_026905.wav,0,1,541,40.73033,-73.9987,2019,51,...,1,0,0,0,0,1,2019-12-29 13:00:00,POINT (-73.9987 40.73033),afternoon,32.99
2,test,1,01_026976.wav,0,1,541,40.73033,-73.9987,2019,20,...,1,0,0,0,0,0,2019-05-22 14:00:00,POINT (-73.9987 40.73033),afternoon,31.49
3,test,1,01_026984.wav,0,1,541,40.73033,-73.9987,2019,23,...,0,0,1,0,0,1,2019-06-11 19:00:00,POINT (-73.9987 40.73033),evening,32.54
4,test,1,01_027018.wav,0,1,541,40.73033,-73.9987,2019,31,...,0,0,0,0,1,0,2019-08-07 18:00:00,POINT (-73.9987 40.73033),evening,30.78


In [22]:
print(df_verified['date_time'].min())
print(df_verified['date_time'].max())

2019-04-22 10:00:00
2020-01-05 11:00:00


In [23]:
df_engine = df_verified[df_verified['1_engine_presence'] == 1]
df_machinery_impact = df_verified[df_verified['2_machinery-impact_presence'] == 1]
df_non_machinery_impact = df_verified[df_verified['3_non-machinery-impact_presence'] == 1]
df_powered_saw = df_verified[df_verified['4_powered-saw_presence'] == 1]
df_alert_signal = df_verified[df_verified['5_alert-signal_presence'] == 1]
df_music = df_verified[df_verified['6_music_presence'] == 1]
df_human_voice = df_verified[df_verified['7_human-voice_presence'] == 1]
df_dog = df_verified[df_verified['8_dog_presence']== 1]

In [28]:
df_verified.to_csv("/Volumes/Extreme SSD/df_verified.csv", index=False)
df_engine.to_csv("/Volumes/Extreme SSD/df_engine.csv", index=False)
df_machinery_impact.to_csv("/Volumes/Extreme SSD/df_machinery_impact.csv", index=False)
df_non_machinery_impact.to_csv("/Volumes/Extreme SSD/df_non_machinery_impact.csv", index=False)
df_powered_saw.to_csv("/Volumes/Extreme SSD/df_powered_saw.csv", index=False)
df_alert_signal.to_csv("/Volumes/Extreme SSD/df_alert_signal.csv", index=False)
df_music.to_csv("/Volumes/Extreme SSD/df_music.csv", index=False)
df_human_voice.to_csv("/Volumes/Extreme SSD/df_human_voice.csv", index=False)
df_dog.to_csv("/Volumes/Extreme SSD/df_dog.csv", index=False)